#PART A — MERGING TABULAR FILES IN GOOGLE COLAB

In [ ]:
# Move to working directory
%cd /content

/content


###Clone the GitHub repository in Colab

In [ ]:
# Clone your GitHub repository
!git clone https://github.com/sandipbarik-bioinfo/Whole-Exome-sequencing-Variant-Analysis.git

Cloning into 'Whole-Exome-sequencing-Variant-Analysis'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 255 (delta 1), reused 9 (delta 1), pack-reused 243 (from 1)
Receiving objects: 100% (255/255), 188.35 MiB | 17.79 MiB/s, done.
Resolving deltas: 100% (108/108), done.
Updating files: 100% (226/226), done.
Filtering content: 100% (111/111), 3.57 GiB | 53.63 MiB/s, done.


###Define the directory containing all extracted and locate the extracted tabular files

In [ ]:
import glob

# tabular variant files from WES analysis
tabular_dir = "/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted"

# sorted() ensures consistent ordering across runs
tabular_files = sorted(glob.glob(f"{tabular_dir}/*.tabular"))

# Print the total number of tabular files found
print("Total tabular files found:", len(tabular_files))
tabular_files[:5]


Total tabular files found: 111


['/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted/SRR8898191.tabular',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted/SRR8898192.tabular',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted/SRR8898193.tabular',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted/SRR8898194.tabular',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/tabular_extracted/SRR8898195.tabular']

In [ ]:
import pandas as pd

# Load one sample tabular file to inspect the structure

df_test = pd.read_csv(tabular_files[0], sep="\t")
df_test.head()


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,DP,AF,GEN[0].GT,GEN[0].AD,GEN[0].GQ,ANN[*].GENE,ANN[*].EFFECT,ANN[*].IMPACT,ANN[*].HGVS_C,ANN[*].HGVS_P,ANN[*].BIOTYPE,LOF,NMD
0,chr1,69511,.,A,G,8341.980,.,338,1.0,1/1,"0,338",.,OR4F5,missense_variant,MODERATE,c.421A>G,p.Thr141Ala,protein_coding,.,.
1,chr1,942451,.,T,C,429.746,.,15,1.0,1/1,"0,15",.,SAMD11,missense_variant,MODERATE,c.505T>C,p.Trp169Arg,protein_coding,.,.
2,chr1,942451,.,T,C,429.746,.,15,1.0,1/1,"0,15",.,SAMD11,missense_variant,MODERATE,c.1030T>C,p.Trp344Arg,protein_coding,.,.
3,chr1,942451,.,T,C,429.746,.,15,1.0,1/1,"0,15",.,SAMD11,missense_variant,MODERATE,c.1027T>C,p.Trp343Arg,protein_coding,.,.
4,chr1,942451,.,T,C,429.746,.,15,1.0,1/1,"0,15",.,SAMD11,missense_variant,MODERATE,c.703T>C,p.Trp235Arg,protein_coding,.,.


###Merge All tabular files

In [ ]:
merged_df_list = []

#  Loop through each tabular file
# - Extract sample ID from filename
# - Read the tabular file
# - Add Sample_ID column
# - Append to the list

for file in tabular_files:
    sample_id = file.split("/")[-1].replace(".tabular", "")
    df = pd.read_csv(file, sep="\t")
    df["Sample_ID"] = sample_id
    merged_df_list.append(df)

# Merge (concatenate) all sample DataFrames into a single combined DataFrame
merged_df = pd.concat(merged_df_list, ignore_index=True)

print("Merged shape:", merged_df.shape)
merged_df.head()


Merged shape: (15792656, 30)


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,DP,AF,GEN[0].GT,...,Sample_ID,AB,GQ,AO,RO,GT,AD,LOF[*].GENE,NMD[*].GENE,version https://git-lfs.github.com/spec/v1
0,chr1,69511.0,.,A,G,8341.980,.,338.0,1.0,1/1,...,SRR8898191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,942451.0,.,T,C,429.746,.,15.0,1.0,1/1,...,SRR8898191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,942451.0,.,T,C,429.746,.,15.0,1.0,1/1,...,SRR8898191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,942451.0,.,T,C,429.746,.,15.0,1.0,1/1,...,SRR8898191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,942451.0,.,T,C,429.746,.,15.0,1.0,1/1,...,SRR8898191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


###Save merged table

In [ ]:
# Save the merged variant dataas a TSV file for downstream analysis

output_file = "Merged_AD_WES_variants.tsv"
merged_df.to_csv(output_file, sep="\t", index=False)

print("Saved:", output_file)


Saved: Merged_AD_WES_variants.tsv


###Basic sanity checks

In [ ]:
# Number of variants per sample
merged_df["Sample_ID"].value_counts().head()

# Check missing values
merged_df.isnull().sum().sort_values(ascending=False).head(10)


,0
version https://git-lfs.github.com/spec/v1,15792652
GT,13556181
RO,13556181
AD,13556181
AB,13556181
AO,13556181
GQ,13556181
LOF[*].GENE,10696244
NMD[*].GENE,10696244
LOF,5096416


###Download of Merged_AD_WES_variants.tsv

In [ ]:
from google.colab import files

files.download('/content/Merged_AD_WES_variants.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for chunk in pd.read_csv('Merged_AD_WES_variants.tsv', sep='\t', chunksize=100000):
    print(chunk.head()) # Processes 100,000 rows at a time

  CHROM       POS ID REF ALT      QUAL FILTER     DP   AF GEN[0].GT  ...  \
0  chr1   69511.0  .   A   G  8341.980      .  338.0  1.0       1/1  ...   
1  chr1  942451.0  .   T   C   429.746      .   15.0  1.0       1/1  ...   
2  chr1  942451.0  .   T   C   429.746      .   15.0  1.0       1/1  ...   
3  chr1  942451.0  .   T   C   429.746      .   15.0  1.0       1/1  ...   
4  chr1  942451.0  .   T   C   429.746      .   15.0  1.0       1/1  ...   

    Sample_ID  AB  GQ  AO  RO  GT  AD LOF[*].GENE NMD[*].GENE  \
0  SRR8898191 NaN NaN NaN NaN NaN NaN         NaN         NaN   
1  SRR8898191 NaN NaN NaN NaN NaN NaN         NaN         NaN   
2  SRR8898191 NaN NaN NaN NaN NaN NaN         NaN         NaN   
3  SRR8898191 NaN NaN NaN NaN NaN NaN         NaN         NaN   
4  SRR8898191 NaN NaN NaN NaN NaN NaN         NaN         NaN   

  version https://git-lfs.github.com/spec/v1  
0                                        NaN  
1                                        NaN  
2          

/tmp/ipython-input-2276229899.py:1: DtypeWarning: Columns (9,10,11,15,22,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('Merged_AD_WES_variants.tsv', sep='\t', chunksize=100000):


       CHROM         POS ID REF   ALT     QUAL FILTER     DP   AF GEN[0].GT  \
800000  chr3  52550415.0  .   T  TCTC  1471.90      .  125.0  0.5       0/1   
800001  chr3  52550415.0  .   T  TCTC  1471.90      .  125.0  0.5       0/1   
800002  chr3  52550415.0  .   T  TCTC  1471.90      .  125.0  0.5       0/1   
800003  chr3  52550415.0  .   T  TCTC  1471.90      .  125.0  0.5       0/1   
800004  chr3  52550771.0  .   T     C  1457.51      .  142.0  0.5       0/1   

        ...   Sample_ID  AB   GQ  AO  RO   GT   AD LOF[*].GENE NMD[*].GENE  \
800000  ...  SRR8898196 NaN  NaN NaN NaN  NaN  NaN         NaN         NaN   
800001  ...  SRR8898196 NaN  NaN NaN NaN  NaN  NaN         NaN         NaN   
800002  ...  SRR8898196 NaN  NaN NaN NaN  NaN  NaN         NaN         NaN   
800003  ...  SRR8898196 NaN  NaN NaN NaN  NaN  NaN         NaN         NaN   
800004  ...  SRR8898196 NaN  NaN NaN NaN  NaN  NaN         NaN         NaN   

       version https://git-lfs.github.com/spec/v1  
8000

/tmp/ipython-input-2276229899.py:1: DtypeWarning: Columns (9,10,11,15,22,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('Merged_AD_WES_variants.tsv', sep='\t', chunksize=100000):


        CHROM         POS ID REF ALT     QUAL FILTER     DP   AF GEN[0].GT  \
1200000  chr2  96328146.0  .   A   G  3981.02      .  291.0  0.5       NaN   
1200001  chr2  96328146.0  .   A   G  3981.02      .  291.0  0.5       NaN   
1200002  chr2  96328146.0  .   A   G  3981.02      .  291.0  0.5       NaN   
1200003  chr2  96328146.0  .   A   G  3981.02      .  291.0  0.5       NaN   
1200004  chr2  96354311.0  .   T   C  4051.39      .  341.0  0.5       NaN   

         ...   Sample_ID        AB GQ     AO     RO GT AD LOF[*].GENE  \
1200000  ...  SRR8898199  0.515464  .  150.0  141.0  .  .         NaN   
1200001  ...  SRR8898199  0.515464  .  150.0  141.0  .  .         NaN   
1200002  ...  SRR8898199  0.515464  .  150.0  141.0  .  .         NaN   
1200003  ...  SRR8898199  0.515464  .  150.0  141.0  .  .         NaN   
1200004  ...  SRR8898199  0.457478  .  156.0  185.0  .  .         NaN   

        NMD[*].GENE version https://git-lfs.github.com/spec/v1  
1200000         NaN        

...

         CHROM         POS ID REF ALT     QUAL FILTER     DP   AF GEN[0].GT  \
15200000  chr6  10775367.0  .   G   A  5752.05      .  393.0  0.5       0/1   
15200001  chr6  10775367.0  .   G   A  5752.05      .  393.0  0.5       0/1   
15200002  chr6  10775367.0  .   G   A  5752.05      .  393.0  0.5       0/1   
15200003  chr6  10775367.0  .   G   A  5752.05      .  393.0  0.5       0/1   
15200004  chr6  10775367.0  .   G   A  5752.05      .  393.0  0.5       0/1   

          ...   Sample_ID  AB  GQ  AO  RO  GT  AD LOF[*].GENE NMD[*].GENE  \
15200000  ...  SRR8898297 NaN NaN NaN NaN NaN NaN           .           .   
15200001  ...  SRR8898297 NaN NaN NaN NaN NaN NaN           .           .   
15200002  ...  SRR8898297 NaN NaN NaN NaN NaN NaN           .           .   
15200003  ...  SRR8898297 NaN NaN NaN NaN NaN NaN           .           .   
15200004  ...  SRR8898297 NaN NaN NaN NaN NaN NaN           .           .   

         version https://git-lfs.github.com/spec/v1  
15200000

/tmp/ipython-input-2276229899.py:1: DtypeWarning: Columns (18,19,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('Merged_AD_WES_variants.tsv', sep='\t', chunksize=100000):


          CHROM         POS ID REF ALT     QUAL FILTER     DP   AF GEN[0].GT  \
15300000  chr19  42935542.0  .   G   A  21135.6      .  640.0  1.0       1/1   
15300001  chr19  43015210.0  .   C   T  14782.9      .  452.0  1.0       1/1   
15300002  chr19  43015210.0  .   C   T  14782.9      .  452.0  1.0       1/1   
15300003  chr19  43015210.0  .   C   T  14782.9      .  452.0  1.0       1/1   
15300004  chr19  43015210.0  .   C   T  14782.9      .  452.0  1.0       1/1   

          ...   Sample_ID  AB  GQ  AO  RO  GT  AD LOF[*].GENE NMD[*].GENE  \
15300000  ...  SRR8898298 NaN NaN NaN NaN NaN NaN         NaN         NaN   
15300001  ...  SRR8898298 NaN NaN NaN NaN NaN NaN         NaN         NaN   
15300002  ...  SRR8898298 NaN NaN NaN NaN NaN NaN         NaN         NaN   
15300003  ...  SRR8898298 NaN NaN NaN NaN NaN NaN         NaN         NaN   
15300004  ...  SRR8898298 NaN NaN NaN NaN NaN NaN         NaN         NaN   

         version https://git-lfs.github.com/spec/v1  
15

/tmp/ipython-input-2276229899.py:1: DtypeWarning: Columns (27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('Merged_AD_WES_variants.tsv', sep='\t', chunksize=100000):


         CHROM          POS ID REF ALT  QUAL FILTER      DP   AF GEN[0].GT  \
15500000  chr7  100958241.0  .   A   C   0.0      .  1119.0  0.0       0/0   
15500001  chr7  100958241.0  .   A   C   0.0      .  1119.0  0.0       0/0   
15500002  chr7  100958244.0  .   C   A   0.0      .  1122.0  0.0       0/0   
15500003  chr7  100958244.0  .   C   A   0.0      .  1122.0  0.0       0/0   
15500004  chr7  100958244.0  .   C   A   0.0      .  1122.0  0.0       0/0   

          ...   Sample_ID  AB  GQ  AO  RO  GT  AD LOF[*].GENE NMD[*].GENE  \
15500000  ...  SRR8898299 NaN NaN NaN NaN NaN NaN           .           .   
15500001  ...  SRR8898299 NaN NaN NaN NaN NaN NaN           .           .   
15500002  ...  SRR8898299 NaN NaN NaN NaN NaN NaN           .           .   
15500003  ...  SRR8898299 NaN NaN NaN NaN NaN NaN           .           .   
15500004  ...  SRR8898299 NaN NaN NaN NaN NaN NaN           .           .   

         version https://git-lfs.github.com/spec/v1  
15500000      

#PART B — MERGING VCF FILES

###Locate all VCF files

In [ ]:
import glob

# vcf variant files from WES analysis
vcf_dir = "/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered"
#sorted() ensures consistent ordering across runs
vcf_files = sorted(glob.glob(f"{vcf_dir}/*.vcf"))

#Print the total number of vcf files and preview the first few file
print("Total VCF files found:", len(vcf_files))
vcf_files[:5]


Total VCF files found: 111


['/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898191_DP10_impact.vcf',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898192_DP10_impact.vcf',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898193_DP10_impact.vcf',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898194_DP10_impact.vcf',
 '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898195_DP10_impact.vcf']

###INSTALL TOOLS ---bcftools and tabix

In [ ]:
# Install bcftools and tabix (which includes bgzip)
!apt-get update
!apt-get install -y bcftools tabix

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,894 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
G

In [ ]:
# Verify installation

!bcftools --version
!tabix --version
!bgzip --version

bcftools 1.13
Using htslib 1.13+ds
Copyright (C) 2021 Genome Research Ltd.
License Expat: The MIT/Expat license
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
tabix (htslib) 1.13+ds
Copyright (C) 2021 Genome Research Ltd.
bgzip (htslib) 1.13+ds
Copyright (C) 2021 Genome Research Ltd.


###bgzip-compress ALL VCF files

This is mandatory — bcftools merge cannot safely merge many uncompressed VCFs.
*   **os**         → file and path handling
*   **subprocess** → run shell commands (bgzip, bcftools)
*   **bgzip** is required for indexing and merging with bcftools

In [ ]:
# Import required libraries

import os
import subprocess
# Initialize an empty list to store bgzip-compressed VCF file paths

gz_vcf_files = []

# Compress each VCF file using bgzip
# bgzip is required for indexing and merging with bcftools
for vcf in vcf_files:
    gz_vcf = vcf + ".gz"
    subprocess.run(f"bgzip -c {vcf} > {gz_vcf}", shell=True)
    gz_vcf_files.append(gz_vcf)

print("Compression completed")


Compression completed


In [ ]:
#Print list of compressed VCF files
print(gz_vcf_files)

['/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898191_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898192_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898193_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898194_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898195_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898196_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898197_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898198_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898199_DP10_impact.vcf.gz', '/content/Whole-Exome-sequencing-Variant-Analysis/data/vcf_filtered/SRR8898200_DP10_impact.vcf.gz',

###Index ALL compressed VCFs
Also mandatory to avoid chromosome order errors.
Creates .tbi index files (required internally)

In [ ]:
# Index each compressed VCF using bcftools

for gz_vcf in gz_vcf_files:
    subprocess.run(f"bcftools index -f {gz_vcf}", shell=True)

print("Indexing completed")


Indexing completed


###Create a VCF list file
his file will be used as input for bcftools merge

In [ ]:
# Create a text file listing all compressed VCFs

vcf_list_file = "vcf_list.txt"

with open(vcf_list_file, "w") as f:
    for vcf in gz_vcf_files:
        f.write(vcf + "\n")

print("VCF list file created")


VCF list file created


###Merge VCFs using --force-samples
Because: 1.   All samples are named 'unknown' 2. Each VCF = one individual 3. Filenames preserve identity
* --force-samples → allows identical sample names
* -m none         → do not merge multiallelic sites
* -l              → input list of VCF files
* -Ov             → output in uncompressed VCF format

In [ ]:
# Merge all indexed VCF files into a single VCF

merged_vcf = "AD_Cohort_111_merged.vcf"

!bcftools merge \
    --force-samples \
    -m none \
    -l vcf_list.txt \
    -Ov \
    -o {merged_vcf}


In [ ]:
# View header
!bcftools view -h AD_Cohort_111_merged.vcf | head -n 40

# Basic stats
!bcftools stats AD_Cohort_111_merged.vcf | head -n 20


##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##fileDate=20260126
##source=freeBayes v1.3.10
##reference=/cvmfs/data.galaxyproject.org/byhand/hg38/sam_index/hg38.fa
##contig=<ID=chr1,length=248956422>
##contig=<ID=chr1_KI270706v1_random,length=175055>
##contig=<ID=chr1_KI270707v1_random,length=32032>
##contig=<ID=chr1_KI270708v1_random,length=127682>
##contig=<ID=chr1_KI270709v1_random,length=66860>
##contig=<ID=chr1_KI270710v1_random,length=40176>
##contig=<ID=chr1_KI270711v1_random,length=42210>
##contig=<ID=chr1_KI270712v1_random,length=176043>
##contig=<ID=chr1_KI270713v1_random,length=40745>
##contig=<ID=chr1_KI270714v1_random,length=41717>
##contig=<ID=chr1_GL383518v1_alt,length=182439>
##contig=<ID=chr1_GL383519v1_alt,length=110268>
##contig=<ID=chr1_GL383520v2_alt,length=366580>
##contig=<ID=chr1_KI270759v1_alt,length=425601>
##contig=<ID=chr1_KI270760v1_alt,length=109528>
##contig=<ID=chr1_KI270761v1_alt,length=165834>
##contig=<ID=chr1_KI270762v1_al

###Download of AD_Cohort_111_merged.vcf

In [ ]:
from google.colab import files

files.download('AD_Cohort_111_merged.vcf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>